### Подготовка данных к задаче прогнозирования

In [1]:
import pandas as pd
import numpy as np

In [2]:
DATE_COL = "rep_date"
COUNTRY_COL = "country"
HS_COL = "hs"
TARGET_COL = "value"

STATIC_COUNTRY_COLS = [
    "distw",
    "unfriendly",
    "brics",
    "cis"
]

COUNTRY_TIME_COLS = [
    "sanctions_proxy",
    "sanctions_proxy_smooth",
    "cpi_yoy",
    "ip_yoy",
    "ex_yoy",
    "logistics_exposure_distw"
]

GLOBAL_TIME_COLS = [
    "gscpi",
    "cb_full_import_value",
    "usd_rub",
    "gov_earn_from_import_nds",
    "gov_expendings"
]

OPTIONAL_EXISTING_COLS = [
    "post_sanctions",
    "unfriendly_post",
    "brics_post",
    "cis_post"
]

SANCTIONS_START = pd.Timestamp("2022-02-01")

In [3]:
def validate_and_clean_base_panel(
    df: pd.DataFrame,
    date_col: str = DATE_COL,
    country_col: str = COUNTRY_COL,
    hs_col: str = HS_COL,
    target_col: str = TARGET_COL,
    static_country_cols: list = STATIC_COUNTRY_COLS,
    country_time_cols: list = COUNTRY_TIME_COLS,
    global_time_cols: list = GLOBAL_TIME_COLS
) -> pd.DataFrame:
    """
    Приводит типы, проверяет обязательные колонки, агрегирует возможные дубликаты
    на уровне rep_date-country-hs и сортирует панель.
    """
    data = df.copy()

    required_cols = (
        [date_col, country_col, hs_col, target_col]
        + static_country_cols
        + country_time_cols
        + global_time_cols
    )

    missing_cols = [c for c in required_cols if c not in data.columns]
    if missing_cols:
        raise ValueError(f"Отсутствуют обязательные колонки: {missing_cols}")

    # Дата
    data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
    if data[date_col].isna().any():
        raise ValueError(f"В {date_col} есть некорректные даты.")

    data[date_col] = data[date_col].dt.to_period("M").dt.to_timestamp()

    # Country / HS
    data[country_col] = data[country_col].astype(str).str.strip()
    data[hs_col] = (
        data[hs_col]
        .astype(str)
        .str.extract(r"(\d+)")[0]
        .str.zfill(4)
    )

    # Numeric
    numeric_cols = [target_col] + static_country_cols + country_time_cols + global_time_cols
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    if data[target_col].isna().any():
        raise ValueError(f"В {target_col} есть NaN после приведения к numeric.")

    # Дубликаты
    dup_mask = data.duplicated(subset=[date_col, country_col, hs_col], keep=False)
    dup_n = dup_mask.sum()

    if dup_n > 0:
        print(f"Найдено {dup_n} строк-дубликатов на уровне rep_date-country-hs. Выполняю агрегацию.")

        agg_dict = {target_col: "sum"}

        for col in static_country_cols + country_time_cols + global_time_cols:
            agg_dict[col] = "first"

        data = (
            data.groupby([date_col, country_col, hs_col], as_index=False)
                .agg(agg_dict)
        )

    data = data.sort_values([date_col, country_col, hs_col]).reset_index(drop=True)
    return data

In [4]:
def check_feature_consistency(
    df: pd.DataFrame,
    date_col: str = DATE_COL,
    country_col: str = COUNTRY_COL,
    hs_col: str = HS_COL,
    static_country_cols: list = STATIC_COUNTRY_COLS,
    country_time_cols: list = COUNTRY_TIME_COLS,
    global_time_cols: list = GLOBAL_TIME_COLS
):
    """
    Проверяет, что признаки действительно имеют ожидаемую структуру:
    - static country cols: constant within country
    - country-time cols: constant within date-country
    - global time cols: constant within date
    """
    data = df.copy()

    problems = []

    # 1. Static by country
    for col in static_country_cols:
        tmp = data.groupby(country_col)[col].nunique(dropna=False)
        bad = tmp[tmp > 1]
        if len(bad) > 0:
            problems.append(f"[STATIC COUNTRY] {col}: {len(bad)} countries have >1 distinct value")

    # 2. Country-time by date-country
    for col in country_time_cols:
        tmp = data.groupby([date_col, country_col])[col].nunique(dropna=False)
        bad = tmp[tmp > 1]
        if len(bad) > 0:
            problems.append(f"[COUNTRY-TIME] {col}: {len(bad)} date-country groups have >1 distinct value")

    # 3. Global-time by date
    for col in global_time_cols:
        tmp = data.groupby(date_col)[col].nunique(dropna=False)
        bad = tmp[tmp > 1]
        if len(bad) > 0:
            problems.append(f"[GLOBAL-TIME] {col}: {len(bad)} months have >1 distinct value")

    if problems:
        print("Найдены проблемы согласованности признаков:")
        for p in problems:
            print(" -", p)
    else:
        print("Все проверки согласованности пройдены успешно.")

In [5]:
def complete_monthly_panel(
    df: pd.DataFrame,
    date_col: str = DATE_COL,
    country_col: str = COUNTRY_COL,
    hs_col: str = HS_COL,
    target_col: str = TARGET_COL,
    static_country_cols: list = STATIC_COUNTRY_COLS,
    country_time_cols: list = COUNTRY_TIME_COLS,
    global_time_cols: list = GLOBAL_TIME_COLS
) -> pd.DataFrame:
    """
    Достраивает полную панель month × country × hs.
    value заполняется нулем.
    Остальные признаки подтягиваются по их логическому уровню.
    """
    data = df.copy()

    all_months = pd.date_range(data[date_col].min(), data[date_col].max(), freq="MS")
    all_countries = np.sort(data[country_col].unique())
    all_hs = np.sort(data[hs_col].unique())

    full_index = pd.MultiIndex.from_product(
        [all_months, all_countries, all_hs],
        names=[date_col, country_col, hs_col]
    )

    full_panel = pd.DataFrame(index=full_index).reset_index()

    # target
    value_df = data[[date_col, country_col, hs_col, target_col]].copy()

    # static country
    static_df = (
        data.groupby(country_col, as_index=False)[static_country_cols]
            .agg("first")
    )

    # country-time
    country_time_df = (
        data.groupby([date_col, country_col], as_index=False)[country_time_cols]
            .agg("first")
    )

    # global-time
    global_time_df = (
        data.groupby(date_col, as_index=False)[global_time_cols]
            .agg("first")
    )

    full_panel = full_panel.merge(value_df, on=[date_col, country_col, hs_col], how="left")
    full_panel = full_panel.merge(static_df, on=country_col, how="left")
    full_panel = full_panel.merge(country_time_df, on=[date_col, country_col], how="left")
    full_panel = full_panel.merge(global_time_df, on=date_col, how="left")

    full_panel[target_col] = full_panel[target_col].fillna(0.0)

    full_panel = full_panel.sort_values([date_col, country_col, hs_col]).reset_index(drop=True)
    return full_panel

In [6]:
def add_deterministic_time_features(
    df: pd.DataFrame,
    date_col: str = DATE_COL,
    sanctions_start: pd.Timestamp = SANCTIONS_START
) -> pd.DataFrame:
    """
    Добавляет признаки, которые известны заранее и не создают утечку:
    календарь, тренд, post-santions, гармоники сезонности.
    """
    data = df.copy()

    data["year"] = data[date_col].dt.year
    data["month"] = data[date_col].dt.month
    data["quarter"] = data[date_col].dt.quarter

    min_date = data[date_col].min()
    data["t"] = (
        (data[date_col].dt.year - min_date.year) * 12
        + (data[date_col].dt.month - min_date.month)
    ).astype(int)

    data["post_sanctions"] = (data[date_col] >= sanctions_start).astype(int)

    data["months_since_shock"] = np.where(
        data[date_col] >= sanctions_start,
        (
            (data[date_col].dt.year - sanctions_start.year) * 12
            + (data[date_col].dt.month - sanctions_start.month)
        ).astype(int),
        0
    )

    data["month_sin_12"] = np.sin(2 * np.pi * data["month"] / 12)
    data["month_cos_12"] = np.cos(2 * np.pi * data["month"] / 12)

    data["series_id"] = data["country"].astype(str) + "__" + data["hs"].astype(str)

    # deterministic interactions
    data["unfriendly_post"] = data["unfriendly"] * data["post_sanctions"]
    data["brics_post"] = data["brics"] * data["post_sanctions"]
    data["cis_post"] = data["cis"] * data["post_sanctions"]

    return data

In [7]:
def make_realistic_supervised_base(
    df: pd.DataFrame,
    horizon: int = 1,
    date_col: str = DATE_COL,
    target_col: str = TARGET_COL
) -> pd.DataFrame:
    """
    Формирует базовую supervised-таблицу:
    признаки на дату t -> target на дату t+h.
    Пока без лагов, только правильная постановка задачи.
    """
    data = df.copy()
    data = data.sort_values(["series_id", date_col]).reset_index(drop=True)

    data["value_now"] = data[target_col]

    data[f"target_h{horizon}"] = (
        data.groupby("series_id")[target_col]
            .shift(-horizon)
    )

    data[f"target_date_h{horizon}"] = data[date_col] + pd.offsets.MonthBegin(horizon)
    data[f"is_usable_h{horizon}"] = data[f"target_h{horizon}"].notna().astype(int)

    return data

In [8]:
def get_realistic_feature_sets():
    """
    Явно фиксирует, какие признаки можно использовать contemporaneously,
    а какие нельзя в реалистичном forecasting.
    """
    allowed_contemporaneous = [
        # static country
        "distw", "unfriendly", "brics", "cis",

        # deterministic calendar/time
        "year", "month", "quarter", "t",
        "post_sanctions", "months_since_shock",
        "month_sin_12", "month_cos_12",

        # deterministic interactions
        "unfriendly_post", "brics_post", "cis_post",

        # identifiers
        "country", "hs", "series_id"
    ]

    forbidden_future_contemporaneous = [
        # country-time
        "sanctions_proxy",
        "sanctions_proxy_smooth",
        "cpi_yoy",
        "ip_yoy",
        "ex_yoy",
        "logistics_exposure_distw",

        # global-time
        "gscpi",
        "cb_full_import_value",
        "usd_rub",
        "gov_earn_from_import_nds",
        "gov_expendings"
    ]

    return {
        "allowed_contemporaneous": allowed_contemporaneous,
        "forbidden_future_contemporaneous": forbidden_future_contemporaneous
    }

In [9]:
def diagnostic_summary(
    df: pd.DataFrame,
    horizon: int = 1,
    date_col: str = DATE_COL,
    target_col: str = TARGET_COL
):
    usable_col = f"is_usable_h{horizon}"
    future_target_col = f"target_h{horizon}"

    print("=" * 80)
    print("DIAGNOSTIC SUMMARY")
    print("=" * 80)
    print(f"Период данных: {df[date_col].min().date()} — {df[date_col].max().date()}")
    print(f"Число строк: {len(df):,}")
    print(f"Число стран: {df['country'].nunique():,}")
    print(f"Число HS4: {df['hs'].nunique():,}")
    print(f"Число рядов country×hs: {df['series_id'].nunique():,}")
    print(f"Доля нулей в текущем value: {(df[target_col] == 0).mean():.3f}")

    if future_target_col in df.columns:
        print(f"Доля usable строк для horizon={horizon}: {df[usable_col].mean():.3f}")
        print(f"Доля нулей в target_h{horizon}: {(df[future_target_col] == 0).mean():.3f}")

    feat_sets = get_realistic_feature_sets()

    print("\nALLOWED CONTEMPORANEOUS FEATURES:")
    print(feat_sets["allowed_contemporaneous"])

    print("\nFORBIDDEN FUTURE CONTEMPORANEOUS FEATURES:")
    print(feat_sets["forbidden_future_contemporaneous"])

In [10]:
df = pd.read_excel('data/final_forecasting_data.xlsx')

panel = validate_and_clean_base_panel(df)
check_feature_consistency(panel)

panel = add_deterministic_time_features(panel)
panel_h1 = make_realistic_supervised_base(panel, horizon=1)

diagnostic_summary(panel_h1, horizon=1)
panel_h1.head()

Все проверки согласованности пройдены успешно.
DIAGNOSTIC SUMMARY
Период данных: 2019-01-01 — 2025-06-01
Число строк: 48,438
Число стран: 69
Число HS4: 9
Число рядов country×hs: 621
Доля нулей в текущем value: 0.553
Доля usable строк для horizon=1: 0.987
Доля нулей в target_h1: 0.547

ALLOWED CONTEMPORANEOUS FEATURES:
['distw', 'unfriendly', 'brics', 'cis', 'year', 'month', 'quarter', 't', 'post_sanctions', 'months_since_shock', 'month_sin_12', 'month_cos_12', 'unfriendly_post', 'brics_post', 'cis_post', 'country', 'hs', 'series_id']

FORBIDDEN FUTURE CONTEMPORANEOUS FEATURES:
['sanctions_proxy', 'sanctions_proxy_smooth', 'cpi_yoy', 'ip_yoy', 'ex_yoy', 'logistics_exposure_distw', 'gscpi', 'cb_full_import_value', 'usd_rub', 'gov_earn_from_import_nds', 'gov_expendings']


,rep_date,country,hs,value,sanctions_proxy,sanctions_proxy_smooth,cpi_yoy,ip_yoy,ex_yoy,gscpi,...,quarter,t,months_since_shock,month_sin_12,month_cos_12,series_id,value_now,target_h1,target_date_h1,is_usable_h1
0,2019-01-01,Albania,9018,0.0,0,0,1.930000,5.8,-0.163976,0.515423,...,1,0,0,0.500000,8.660254e-01,Albania__9018,0.0,0.0,2019-02-01,1
1,2019-02-01,Albania,9018,0.0,0,0,1.716270,5.8,2.086924,0.145187,...,1,1,0,0.866025,5.000000e-01,Albania__9018,0.0,0.0,2019-03-01,1
2,2019-03-01,Albania,9018,0.0,0,0,1.107924,5.8,3.863434,0.216385,...,1,2,0,1.000000,6.123234e-17,Albania__9018,0.0,0.0,2019-04-01,1
3,2019-04-01,Albania,9018,0.0,0,0,1.351889,5.9,5.080519,0.017049,...,2,3,0,0.866025,-5.000000e-01,Albania__9018,0.0,0.0,2019-05-01,1
4,2019-05-01,Albania,9018,0.0,0,0,1.529021,6.0,2.264640,-0.623563,...,2,4,0,0.500000,-8.660254e-01,Albania__9018,0.0,0.0,2019-06-01,1


In [11]:
panel_h1.to_excel('data/prepared_forecasting_data.xlsx', index=False)

В прогнозной части исследования используется реалистичная постановка задачи, при которой модель предсказывает месячный объем импорта медицинского оборудования в денежном выражении на уровне пары «страна-экспортер × товарная группа HS4» на горизонт h месяцев вперед. Базовой единицей наблюдения является monthly panel rep_date × country × hs.

На первом этапе признаки были разделены по их информационной структуре. К статическим признакам страны были отнесены расстояние до России и индикаторы принадлежности к отдельным страновым блокам. К country-time переменным были отнесены санкционный прокси, его сглаженная версия, макроэкономические показатели стран-экспортеров и переменная логистического воздействия. Отдельную группу составили global time-only переменные, одинаковые для всех наблюдений внутри одного месяца, включая глобальный индекс логистического давления, совокупный импорт Российской Федерации, курс доллара к рублю, доходы бюджета от импортного НДС и общий объем государственных расходов. Кроме того, были сформированы детерминированные временные признаки, такие как календарный месяц, квартал, линейный тренд, индикатор постсанкционного периода и гармонические компоненты сезонности.

Принципиальным условием реалистичного прогнозирования является отсутствие утечки информации из будущего. Поэтому динамические country-time и global time-only переменные не используются в contemporaneous форме для прогнозируемого периода, поскольку их фактические значения в момент построения прогноза неизвестны. На первом этапе они сохраняются в базе данных, но в качестве входов моделей могут использоваться только на следующих шагах через лаговые преобразования, rolling statistics или сценарные траектории. После этого для каждого наблюдения на дату t была сформирована целевая переменная target_h, отражающая объем импорта в период t+h, что позволило задать корректную supervised-постановку задачи прогнозирования.